# N153 · SQL聚合、连接与计数语义

**目标：** 避免连接放大和错误计数。

**先修：** N152。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** GROUP BY/HAVING；COUNT(*)/COUNT(col)；INNER/LEFT/SELF JOIN；多对多。

## 从问题到算法

连接会改变行数。LEFT JOIN保留没有地址的人，但若把右表条件写进WHERE，可能又排除NULL行。自连接要用不同别名区分员工与经理。聚合时COUNT(*)计算行，COUNT(col)忽略NULL；经理是否有五个下属不能按非空工资数计。

## 最小实现

**本章接口：** `query_people_addresses.sql`、`query_employees_gt_manager.sql`、`query_managers.sql`

In [1]:
import sqlite3

SQL_QUERIES = {
    'query_people_addresses.sql': r'''SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p LEFT JOIN Address AS a ON a.personId = p.personId;''',
    'query_employees_gt_manager.sql': r'''SELECT e.name AS Employee
FROM Employee AS e JOIN Employee AS m ON e.managerId = m.id
WHERE e.salary > m.salary;''',
    'query_managers.sql': r'''SELECT m.name
FROM Employee AS e JOIN Employee AS m ON e.managerId = m.id
GROUP BY m.id, m.name
HAVING COUNT(*) >= 5;''',
}

def create_sqlite_fixture():
    connection=sqlite3.connect(':memory:')
    connection.executescript(r'''
CREATE TABLE Person(personId INTEGER PRIMARY KEY,firstName TEXT,lastName TEXT);
CREATE TABLE Address(addressId INTEGER PRIMARY KEY,personId INTEGER,city TEXT,state TEXT);
INSERT INTO Person VALUES (1,'Ann','A'),(2,'Ben','B');
INSERT INTO Address VALUES (1,1,'CityA','StateA');
CREATE TABLE Employee(id INTEGER PRIMARY KEY,name TEXT,salary INTEGER,managerId INTEGER);
INSERT INTO Employee VALUES (1,'Boss',100,NULL),(2,'E2',120,1),(3,'E3',80,1),(4,'E4',NULL,1),(5,'E5',90,1),(6,'E6',110,1),(7,'NoManager',200,NULL);
''')
    return connection

def query_rows(connection,name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 正确性、前提与复杂度

左连接为每个左行枚举匹配右行，无匹配时补NULL；一对多连接会复制左行，不能默认输出仍是一人一行。GROUP BY以经理ID区分同名经理，HAVING在分组之后筛选，COUNT(*)恰计下属记录。工资为NULL的大小关系不为TRUE，不会误判为高薪员工。

**代价：** 连接代价依索引与查询计划变化；不把SQL短就当作O(1)。无索引嵌套连接可达O(NM)。本地测试使用SQLite、明确主键和重复地址的一对多边界。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture(); cursor=db.execute(SQL_QUERIES['query_people_addresses.sql'])
show_table([x[0] for x in cursor.description],cursor.fetchall())
show_table(['员工行数','非空工资数'],db.execute('SELECT COUNT(*), COUNT(salary) FROM Employee').fetchall())
db.close()

firstName,lastName,city,state
Ann,A,CityA,StateA
Ben,B,None,None


员工行数,非空工资数
7,6


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
db=create_sqlite_fixture()
assert set(query_rows(db,'query_people_addresses.sql'))=={('Ann','A','CityA','StateA'),('Ben','B',None,None)}
assert set(query_rows(db,'query_employees_gt_manager.sql'))=={('E2',),('E6',)}
assert query_rows(db,'query_managers.sql')==[('Boss',)]
assert db.execute('SELECT COUNT(*), COUNT(salary) FROM Employee WHERE managerId=1').fetchone()==(5,4)
db.execute('DELETE FROM Employee WHERE id=6'); assert query_rows(db,'query_managers.sql')==[]
db.execute("INSERT INTO Address VALUES (2,1,'CityB','StateB')")
assert len(query_rows(db,'query_people_addresses.sql'))==3
db.close()
print('N153: 所有本章断言通过')

N153: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造一对多连接使SUM重复的例子。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分WHERE和HAVING。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 175. Combine Two Tables（canonical）
- 181. Employees Earning More Than Their Managers（canonical）
- 570. Managers with at Least 5 Direct Reports（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。